# Task 1: Model Training and Optimization Pipeline
Use this notebook to perform your data preprocessing, hyperparameter tuning via Cross-Validation, and final evaluation on the test set.

In [5]:
import pandas as pd
import numpy as np
import pickle
import time
import optuna
import trackio
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder
from scipy.stats import randint
import json
import os
import joblib

# Add any other imports you need here

## 1. Data Loading & Preprocessing
Load `train.csv` and `test.csv`. Convert string categorical variables to numeric.
**Required:** Save your label encoders/mappings because your Streamlit UI will need them later to prepare user inputs for inference!

In [6]:
# TODO: Implement your preprocessing here (use LabelEncoder or manual dictionaries)
# Ensure you keep all necessary features that will be shown on the UI dashboard.


# TODO: Separate predictors (X) and target (y: 'price')
# Load the datasets
train_df = pd.read_csv('Dataset/train.csv')
test_df = pd.read_csv('Dataset/test.csv')

categorical_cols = ['city', 'location', 'Status', 'property_type']
frontend_mappings = {}

for col in categorical_cols:
    le = LabelEncoder()
    
    # 1. Combine train and test categories to fit the encoder on ALL known labels
    all_categories = pd.concat([train_df[col], test_df[col]]).astype(str)
    le.fit(all_categories)
    
    # 2. Transform both datasets safely
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))
    
    # 3. Extract the mapping for the frontend
    frontend_mappings[col] = {category: int(index) for index, category in enumerate(le.classes_)}

# Save the mappings
with open('frontend_mappings.json', 'w') as json_file:
    json.dump(frontend_mappings, json_file, indent=4)

print("Categorical mappings successfully saved to 'frontend_mappings.json'.")

# Separate predictors (X) and target (y: 'price') for the training set
X_train = train_df.drop(columns=['price'])
y_train = train_df['price']

# If the test set includes the target variable, separate it as well
if 'price' in test_df.columns:
    X_test = test_df.drop(columns=['price'])
    y_test = test_df['price']
else:
    X_test = test_df

print("Predictors (X) and target (y) separated.")

Categorical mappings successfully saved to 'frontend_mappings.json'.
Predictors (X) and target (y) separated.


## 2. Hyperparameter Tuning using Cross-Validation

**Strict Search Space:**
- `n_estimators`: 50 to 200
- `max_depth`: 10 to 30
- `min_samples_split`: 2 to 10

Implement Grid Search, Random Search, and Bayesian Optimization (using Optuna). Evaluate each using 5-fold cross-validation on `train_df`.

In [7]:
import time
import pandas as pd
import optuna

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
import trackio 

# Assuming train_df is already loaded and encoded from your previous pipeline
X_train = train_df.drop(columns=['price'])
y_train = train_df['price']

rf = RandomForestRegressor(random_state=42)

# Initialize trackio project/experiment here
trackio.init(project="sttai-assingment4")

# ==========================================
# 1. Grid Search Implementation
# ==========================================
print("Starting Grid Search...")
start_time_gs = time.time()

# Grid Search requires discrete lists. We step through the strict search space.
param_grid = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [10, 15, 20, 25, 30],
    'min_samples_split': [2, 4, 6, 8, 10]
}

grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='r2', 
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

gs_time = time.time() - start_time_gs
gs_iterations = len(grid_search.cv_results_['params'])
gs_best_score = grid_search.best_score_
gs_best_params = grid_search.best_params_

# Log to trackio using the correct syntax
trackio.log(
    metrics={
        "method": "GridSearchCV",
        "time_taken_seconds": float(gs_time),
        "iterations": int(gs_iterations),
        "best_cv_score": float(gs_best_score),
        "best_n_estimators": int(gs_best_params['n_estimators']),
        "best_max_depth": int(gs_best_params['max_depth']),
        "best_min_samples_split": int(gs_best_params['min_samples_split'])
    }
)
print(f"Grid Search Complete | Best Score: {gs_best_score:.4f} | Time: {gs_time:.2f}s")


# ==========================================
# 2. Random Search Implementation
# ==========================================
print("\nStarting Random Search...")
start_time_rs = time.time()

# Random search samples from distributions within your strict bounds
param_distributions = {
    'n_estimators': randint(50, 201), 
    'max_depth': randint(10, 31),
    'min_samples_split': randint(2, 11)
}

rs_iterations = 50

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_distributions,
    n_iter=rs_iterations,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

rs_time = time.time() - start_time_rs
rs_best_score = random_search.best_score_
rs_best_params = random_search.best_params_

# Log to trackio using the correct syntax
trackio.log(
    metrics={
        "method": "RandomizedSearchCV",
        "time_taken_seconds": float(rs_time),
        "iterations": int(rs_iterations),
        "best_cv_score": float(rs_best_score),
        "best_n_estimators": int(rs_best_params['n_estimators']),
        "best_max_depth": int(rs_best_params['max_depth']),
        "best_min_samples_split": int(rs_best_params['min_samples_split'])
    }
)
print(f"Random Search Complete | Best Score: {rs_best_score:.4f} | Time: {rs_time:.2f}s")


# ==========================================
# 3. Bayesian Optimization (Optuna) Implementation
# ==========================================
print("\nStarting Bayesian Optimization (Optuna)...")
start_time_optuna = time.time()
optuna_iterations = 50

def objective(trial):
    # Enforce strict search space
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 10, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'random_state': 42
    }
    
    model = RandomForestRegressor(**params)
    
    # 5-fold Cross-Validation
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2', n_jobs=-1)
    return scores.mean()

optuna.logging.set_verbosity(optuna.logging.WARNING) 
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=optuna_iterations)

optuna_time = time.time() - start_time_optuna
optuna_best_score = study.best_value
optuna_best_params = study.best_params

# Log to trackio using the correct syntax
trackio.log(
    metrics={
        "method": "Optuna_BayesianOptimization",
        "time_taken_seconds": float(optuna_time),
        "iterations": int(optuna_iterations),
        "best_cv_score": float(optuna_best_score),
        "best_n_estimators": int(optuna_best_params['n_estimators']),
        "best_max_depth": int(optuna_best_params['max_depth']),
        "best_min_samples_split": int(optuna_best_params['min_samples_split'])
    }
)

trackio.finish()
print(f"Optuna Complete | Best Score: {optuna_best_score:.4f} | Time: {optuna_time:.2f}s")

# ==========================================
# 4. Save Search Objects to Disk
# ==========================================
import joblib
import os
os.makedirs('models', exist_ok=True)
joblib.dump(grid_search, 'models/grid_search.pkl')
joblib.dump(random_search, 'models/random_search.pkl')
joblib.dump(study, 'models/optuna_study.pkl')
print("\nAll cross-validation search objects saved to 'models/' successfully!")


* Trackio project initialized: sttai-assingment4
* Trackio metrics logged to: C:\Users\harsh\.cache\huggingface\trackio
* NVIDIA GPU detected, enabling automatic GPU metrics logging
* Created new run: dainty-sunset-0


Starting Grid Search...
Grid Search Complete | Best Score: 0.9854 | Time: 366.31s

Starting Random Search...
Random Search Complete | Best Score: 0.9854 | Time: 177.45s

Starting Bayesian Optimization (Optuna)...
* Run finished. Uploading logs to Trackio (please wait...)
Optuna Complete | Best Score: 0.9854 | Time: 359.67s

All cross-validation search objects saved to 'models/' successfully!


In [8]:
trackio.show()

## 3. Evaluation & Plots
Plot the compute trials (iterations) vs. cross-validation error, and plot the hyperparameter space to visualize how the Bayesian method explored the space.

In [8]:
# Try loading from disk if the objects aren't in memory so you don't have to rerun the 20min cell!
import joblib
import os

try:
    if 'grid_search' not in locals():
        grid_search = joblib.load('models/grid_search.pkl')
        print("Loaded grid_search from disk.")
    if 'random_search' not in locals():
        random_search = joblib.load('models/random_search.pkl')
        print("Loaded random_search from disk.")
    if 'study' not in locals():
        study = joblib.load('models/optuna_study.pkl')
        print("Loaded optuna_study from disk.")
except FileNotFoundError:
    print("Could not find saved models on disk. You need to run the CV cell above at least once.")

# TODO: Generate and save trials_vs_error.png
# X-axis: Number of iterations
# Y-axis: Best CV error found so far
# Overlay Grid, Random, and Bayesian methods on the same plot.

# TODO: Generate and save optuna_hyperparameter_space.png

# Using Optuna's matplotlib backend to easily save as PNG
from optuna.visualization.matplotlib import plot_contour

# Ensure the plots directory exists
os.makedirs('plots', exist_ok=True)

# ==========================================
# 1. Generate and save trials_vs_error.png
# ==========================================

# Extract the chronological mean test scores from each method
gs_scores = grid_search.cv_results_['mean_test_score']
rs_scores = random_search.cv_results_['mean_test_score']
optuna_scores = np.array([trial.value for trial in study.trials if trial.value is not None])

# Convert R-squared scores to Error (1 - R2)
gs_errors = 1 - gs_scores
rs_errors = 1 - rs_scores
optuna_errors = 1 - optuna_scores

# Calculate the running best (minimum) error found up to each iteration
gs_best_errors = np.minimum.accumulate(gs_errors)
rs_best_errors = np.minimum.accumulate(rs_errors)
optuna_best_errors = np.minimum.accumulate(optuna_errors)

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(gs_best_errors) + 1), gs_best_errors, label='Grid Search', linewidth=2)
plt.plot(range(1, len(rs_best_errors) + 1), rs_best_errors, label='Random Search', linewidth=2)
plt.plot(range(1, len(optuna_best_errors) + 1), optuna_best_errors, label='Bayesian Optimization', linewidth=2)

plt.title('Compute Budget vs. Best CV Error')
plt.xlabel('Number of Iterations / Trials')
plt.ylabel('Best CV Error Found (1 - R²)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)

# Save Plot 1
plt.savefig('plots/trials_vs_error.png', bbox_inches='tight', dpi=300)
plt.close()
print("Successfully generated and saved: plots/trials_vs_error.png")


# ==========================================
# 2. Generate and save optuna_hyperparameter_space.png
# ==========================================

# Plot a contour map showing how Optuna explored the relationship between three main parameters
axes = plot_contour(study, params=['n_estimators', 'max_depth', 'min_samples_split'])
fig = plt.gcf()
fig.set_size_inches(15, 10)
fig.suptitle('Optuna: Pairwise Parameters')

# Save Plot 2
fig.savefig('plots/optuna_hyperparameter_space.png', bbox_inches='tight', dpi=300)
plt.close()
print("Successfully generated and saved: plots/optuna_hyperparameter_space.png")

Successfully generated and saved: plots/trials_vs_error.png


C:\Users\harsh\AppData\Local\Temp\ipykernel_4500\3972094177.py:73: ExperimentalWarning: optuna.visualization.matplotlib._contour.plot_contour is experimental (supported from v2.2.0). The interface can change in the future.
  axes = plot_contour(study, params=['n_estimators', 'max_depth', 'min_samples_split'])


Successfully generated and saved: plots/optuna_hyperparameter_space.png


## 4. Final Testing & Model Saving
Report the best hyperparameters found, train your overall best model on the entire `train.csv`, evaluate on `test.csv`, and save the model file.

In [9]:
# Ensure the models directory exists
os.makedirs('models', exist_ok=True)


# 1. Report the best hyperparameters


print("=== Hyperparameter Tuning Results ===")
print(f"Grid Search Best Parameters: {grid_search.best_params_} (Score: {gs_best_score:.4f})")
print(f"Random Search Best Parameters: {random_search.best_params_} (Score: {rs_best_score:.4f})")
print(f"Optuna Best Parameters: {study.best_params} (Score: {optuna_best_score:.4f})")

# Determine the overall best method based on CV score
best_scores = {
    'Grid Search': gs_best_score,
    'Random Search': rs_best_score,
    'Optuna': optuna_best_score
}

overall_best_method = max(best_scores, key=best_scores.get)
print(f"\nOverall Best Method: {overall_best_method} with score {best_scores[overall_best_method]:.4f}")

# Extract the absolute best hyperparameters
if overall_best_method == 'Grid Search':
    best_params = grid_search.best_params_
elif overall_best_method == 'Random Search':
    best_params = random_search.best_params_
else:
    best_params = study.best_params

print(f"Using Best Parameters: {best_params}")



# 2. Train the overall best model

print("\nTraining final model on the full training dataset...")
# Instantiate a new model with the best parameters
best_model = RandomForestRegressor(**best_params, random_state=42)

# Train on the entire training set
best_model.fit(X_train, y_train)
print("Training complete.")



# 3. Evaluate on X_test


# X_test and y_test were defined in the initial preprocessing block
if 'price' in test_df.columns:
    print("\nEvaluating on test data...")
    predictions = best_model.predict(X_test)
    
    # Calculate Mean Absolute Error (MAE)
    mae = mean_absolute_error(y_test, predictions)
    print(f"Test Set Mean Absolute Error (MAE): {mae:,.2f}")
    
    # Log final metric to trackio
    trackio.log(
        metrics={
            "test_set_mae": float(mae),
            "final_model_n_estimators": int(best_params['n_estimators']),
            "final_model_max_depth": int(best_params['max_depth']),
            "final_model_min_samples_split": int(best_params['min_samples_split'])
        }
    )
else:
    print("\nNo 'price' column found in test_df. Cannot compute MAE. Generating predictions only.")
    predictions = best_model.predict(X_test)




# Save the trained model
model_path = os.path.join('models', 'best_model.pkl')
joblib.dump(best_model, model_path)
print(f"\nSaved model to: {model_path}")

=== Hyperparameter Tuning Results ===
Grid Search Best Parameters: {'max_depth': 25, 'min_samples_split': 2, 'n_estimators': 200} (Score: 0.9854)
Random Search Best Parameters: {'max_depth': 30, 'min_samples_split': 3, 'n_estimators': 160} (Score: 0.9854)
Optuna Best Parameters: {'n_estimators': 191, 'max_depth': 26, 'min_samples_split': 3} (Score: 0.9854)

Overall Best Method: Optuna with score 0.9854
Using Best Parameters: {'n_estimators': 191, 'max_depth': 26, 'min_samples_split': 3}

Training final model on the full training dataset...
Training complete.

Evaluating on test data...
Test Set Mean Absolute Error (MAE): 1,501.09

Saved model to: models\best_model.pkl
